
# Huggingfaceへのアップロード

## 事前準備

HUGGGING Faceからトークンを発行し、ログインしておく。
(初回のみ)

In [6]:
HUGGINGFACE_TOKEN=""

In [7]:
!huggingface-cli login --token {HUGGINGFACE_TOKEN} --add-to-git-credential

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
Token is valid (permission: write).
The token `jetracer` has been saved to /home/jetson/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Token has not been saved to git credential helper.
Your token has been saved to /home/jetson/.cache/huggingface/token
Login successful.
The current active token is: `jetracer`


## ログの表示用 Widget

In [22]:
import ipywidgets
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label
import os
import glob
from IPython.display import clear_output

l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0
def write_log(msg):
    global process_widget, process_no
    process_no = process_no + 1
    process_widget.value = str(process_no) + ": " + msg + "\n" + process_widget.value
    
    # UIのクリアと更新
    clear_output(wait=True)

In [23]:
import subprocess
import os 
import shutil 
import datetime 
from pathlib import Path

task_dropdown = ipywidgets.Dropdown(options=[], description='タスク')
dataset_dropdown = ipywidgets.Dropdown(options=[], description='データセット')
xy_data_count_widget = ipywidgets.IntText(description='XYデータ数')
speed_data_count_widget = ipywidgets.IntText(description='速度データ数')

dataset_path = ""
base_path = "./"
HOME = "/home/jetson/notebooks/notebooks/"
INPUT_ROOT = Path(HOME+dataset_path)
OUTPUT_DIR = None 
CAMERA_KEY = "observation.images.front"
FPS = 30
EPISODE_SECONDS = None
TASK_DESC = "Jetracer Driving task"
NORMALIZE_FROM_0_224 = True

def update_count(change):
    global base_path, dataset_path, HOME, INPUT_ROOT
    
    task = task_dropdown.value
    dataset = dataset_dropdown.value
    dataset_path = base_path + "/" + task + "/" + dataset
    xy_path = base_path + "/" + task + "/" + dataset + "/xy/"
    speed_path = base_path + "/" + task + "/" + dataset + "/speed/"

    xy_is_dir = os.path.isdir(xy_path)
    speed_is_dir = os.path.isdir(speed_path)

    xy_file_count = 0
    speed_file_count = 0

    if xy_is_dir:
        xy_file_count = sum(os.path.isfile(os.path.join(xy_path,name)) for name in os.listdir(xy_path))
        xy_data_count_widget.value = xy_file_count
    else:
        xy_data_count_widget.value = 0
        
    if speed_is_dir:
        speed_file_count = sum(os.path.isfile(os.path.join(speed_path,name)) for name in os.listdir(speed_path))
        speed_data_count_widget.value = speed_file_count
    else:
        speed_data_count_widget.value = 0
    INPUT_ROOT = Path(HOME+dataset_path)
    write_log(f"データセットを: {INPUT_ROOT}に設定")
        
dataset_dropdown.observe(update_count, names='value')
        
def change_dataset(change):
    global base_path

    if not change['new']:
        write_log("タスクが選択されていません。")
        return
    
    try:
        path = os.path.join(base_path, str(change['new']))
        write_log(f"change_dataset: {path}")
        if not os.path.exists(path):
            write_log(f"{path}が存在していません。")
            return
        
        dirs = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))
                and not f.startswith(".")
                and f not in {"__pycache__"}]
        dirs = sorted(dirs)
        dataset_dropdown.options = dirs
    except Exception as e:
        write_log(f"Error: {str(e)}")
        dataset_dropdown.options = []

def change_task():
    global base_path
    
    if not os.path.exists(base_path):
        write_log(f"{base_path}が存在していません。")
        return
    
    try:
        dirs = [f for f in os.listdir(base_path) 
                if os.path.isdir(os.path.join(base_path, f))
                and not f.startswith(".")
                and f not in {"__pycache__", "model_c", "model", "model_trt", "video", "zip"}]
        dirs = sorted(dirs)
        task_dropdown.options = dirs
    except Exception as e:
        write_log(f"Error: {str(e)}")
        task_dropdown.options = []

task_dropdown.observe(change_dataset, names='value')
change_task()

## 必要なライブラリのImportとユーティリティの設定

In [24]:
from dataclasses import dataclass
from typing import List, Tuple, Dict
import re, json, math
import numpy as np
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import imageio.v3 as iio
from tqdm import tqdm

try:
    import cv2  # optional
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

@dataclass
class FrameRec:
    idx: int
    path: Path
    steering_raw: float
    throttle_raw: float

def ensure_empty_dir(p: Path):
    # Remove and re-create output tree
    if p.exists():
        import shutil
        shutil.rmtree(p)
    (p / "meta").mkdir(parents=True, exist_ok=True)
    (p / "data" / "chunk-000").mkdir(parents=True, exist_ok=True)
    (p / "meta" / "episodes" / "chunk-000").mkdir(parents=True, exist_ok=True)
    (p / "videos" / CAMERA_KEY / "chunk-000").mkdir(parents=True, exist_ok=True)
    (p / "meta").mkdir(parents=True, exist_ok=True)

def parse_xy_from_name(name: str) -> Tuple[int|float, int|float] | None:
    # Extract x and y from filename: any_{x}_{y}.(png|jpg|jpeg)
    base = name.rsplit(".", 1)[0]
    m = re.search(r"_(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)$", base)
    if not m:
        return None
    x = float(m.group(1))
    y = float(m.group(2))
    if x.is_integer(): x = int(x)
    if y.is_integer(): y = int(y)
    return x, y

def find_xy_frames(input_root: Path) -> List[FrameRec]:
    xy_dir = input_root / "xy"
    if not xy_dir.is_dir():
        raise FileNotFoundError(f"xy/ not found: {xy_dir}")
    exts = (".png",".jpg",".jpeg",".bmp",".webp")
    files = sorted([p for p in xy_dir.iterdir() if p.suffix.lower() in exts])
    frames: List[FrameRec] = []
    for i, p in enumerate(files):
        pr = parse_xy_from_name(p.name)
        if pr is None:
            continue
        x, y = pr
        frames.append(FrameRec(idx=i, path=p, steering_raw=x, throttle_raw=y))
    return frames

def to_actions(frames: List[FrameRec]) -> np.ndarray:
    # Map x->steering, y->speed_forward
    if NORMALIZE_FROM_0_224:
        def norm(v): return (float(v) / 112.0) - 1.0
        steer = np.array([norm(fr.steering_raw) for fr in frames], dtype=np.float32)
        speed = np.array([norm(fr.throttle_raw) for fr in frames], dtype=np.float32)
    else:
        steer = np.array([float(fr.steering_raw) for fr in frames], dtype=np.float32)
        speed = np.array([float(fr.throttle_raw) for fr in frames], dtype=np.float32)
    return np.stack([steer, speed], axis=1)  # [N,2]

def load_image(path: Path) -> np.ndarray:
    # RGB ndarray(H,W,3)
    if _HAS_CV2:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    else:
        return iio.imread(path)

def write_video(frames: List[FrameRec], out_path: Path, fps: int):
    # Encode mp4 (libx264, yuv420p)
    import imageio_ffmpeg as ffmpeg
    first = load_image(frames[0].path)
    h, w = int(first.shape[0]), int(first.shape[1])

    cmd = [ffmpeg.get_ffmpeg_exe(), "-y", "-loglevel", "error",
           "-f", "rawvideo", "-vcodec", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-", "-an",
           "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
           str(out_path)]
    import subprocess
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)

    try:
        for fr in tqdm(frames, desc="encode video", unit="frame"):
            img = load_image(fr.path)
            assert img.shape[0]==h and img.shape[1]==w
            proc.stdin.write(img.tobytes(order="C"))
    finally:
        proc.stdin.close()
        proc.wait()

def write_parquet(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, path)

def split_episodes(n_frames: int, episode_seconds: int|None, fps: int) -> List[Tuple[int,int,int]]:
    if episode_seconds is None:
        return [(0, 0, n_frames)]
    L = int(episode_seconds * fps)
    if L <= 0:
        return [(0, 0, n_frames)]
    out = []
    s = 0
    ep = 0
    while s < n_frames:
        t = min(n_frames, s + L)
        out.append((ep, s, t))
        ep += 1
        s = t
    return out

def compute_stats(actions: np.ndarray) -> Dict:
    return {
        "action": {
            "mean": actions.mean(axis=0).tolist(),
            "std":  actions.std(axis=0).tolist(),
            "min":  actions.min(axis=0).tolist(),
            "max":  actions.max(axis=0).tolist(),
        }
    }


## episodes/tasks/info.jsonの出力

In [25]:
def write_tasks_parquet(out_root: Path, task_text: str):
    df = pd.DataFrame({"task_index":[0]}, index=[task_text])
    path = out_root / "meta" / "tasks.parquet"
    df.to_parquet(path)

def write_episodes_parquet(out_root: Path,
                           episodes: List[Tuple[int,int,int]],
                           total_frames: int,
                           fps: int,
                           camera_key: str,
                           video_duration_s: float):
    rows = []
    for (ep_idx, s, t) in episodes:
        from_frame = s
        to_frame   = t
        length     = to_frame - from_frame
        rows.append({
            "episode_index": ep_idx,
            "length": length,
            "data/chunk_index": 0,
            "data/file_index": 0,
            "dataset_from_index": from_frame,
            "dataset_to_index": to_frame,
            f"videos/{camera_key}/chunk_index": 0,
            f"videos/{camera_key}/file_index": 0,
            f"videos/{camera_key}/from_timestamp": (from_frame / float(fps)),
            f"videos/{camera_key}/to_timestamp":   (to_frame   / float(fps)),
            "from_frame_index": from_frame,
            "to_frame_index":   to_frame,
        })
    ep_df = pd.DataFrame(rows)
    path = out_root / "meta" / "episodes" / "chunk-000" / "file-000.parquet"
    write_parquet(ep_df, path)

def write_info_json(out_root: Path,
                    camera_key: str,
                    H: int, W: int,
                    fps: int,
                    total_frames: int,
                    total_episodes: int,
                    codec_str: str = "h264",
                    data_mb: int = 100,
                    video_mb: int = 500):
    info = {
        "codebase_version": "v3.0",
        "robot_type": "unknown",
        "total_episodes": total_episodes,
        "total_frames": total_frames,
        "total_tasks": 1,
        "chunks_size": 1000,
        "fps": int(fps),
        "splits": {"train": f"0:{total_episodes}"},
        "data_path": "data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet",
        "video_path": f"videos/{{video_key}}/chunk-{{chunk_index:03d}}/file-{{file_index:03d}}.mp4",
        "features": {
            camera_key: {
                "dtype": "video",
                "shape": [H, W, 3],
                "names": ["height", "width", "channels"],
                "video_info": {
                    "video.height": H,
                    "video.width": W,
                    "video.channels": 3,
                    "video.codec": codec_str,
                    "video.pix_fmt": "yuv420p",
                    "video.is_depth_map": False,
                    "video.fps": float(fps),
                    "has_audio": False
                },
                "info": {
                    "video.height": H,
                    "video.width": W,
                    "video.channels": 3,
                    "video.codec": codec_str,
                    "video.pix_fmt": "yuv420p",
                    "video.is_depth_map": False,
                    "video.fps": int(fps),
                    "has_audio": False
                }
            },
            "action": {
                "dtype": "float32",
                "shape": [2],
                "names": ["steering", "speed"],
                "fps": int(fps)
            },
            "timestamp": {
                "dtype": "float32",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "frame_index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "episode_index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            },
            "task_index": {
                "dtype": "int64",
                "shape": [1],
                "names": None,
                "fps": int(fps)
            }
        },
        "data_files_size_in_mb": int(data_mb),
        "video_files_size_in_mb": int(video_mb)
    }
    (out_root / "meta").mkdir(parents=True, exist_ok=True)
    with (out_root / "meta" / "info.json").open("w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)


## 変換処理

In [26]:
convert_button = ipywidgets.Button(description='LeRobot形式に変換')
out_dir = ""
def convert(change):
    global out_dir
    if INPUT_ROOT is None or not Path(INPUT_ROOT).exists():
        raise FileNotFoundError(f"INPUT_ROOT does not exist: {INPUT_ROOT}")

    out_dir = OUTPUT_DIR or (INPUT_ROOT / "lerobot_v3_out")
    ensure_empty_dir(out_dir)

    # 1) Read frames & build actions
    frames = find_xy_frames(INPUT_ROOT)
    if len(frames) == 0:
        raise RuntimeError("No {ID_x_y.*} images found under xy/.")

    actions = to_actions(frames)  # [N,2]
    write_log(f"Total frames: {len(frames)}")

    # 2) Image size
    first = load_image(frames[0].path)
    H, W = int(first.shape[0]), int(first.shape[1])
    write_log(f"Image size: {W}x{H}")

    # 3) Episodes
    if EPISODE_SECONDS is None:
        episodes = [(0, 0, len(frames))]
    else:
        episodes = split_episodes(len(frames), EPISODE_SECONDS, FPS)
    total_episodes = len(episodes)
    write_log(f"Episodes: {total_episodes}")

    # 4) Write video at target FPS
    video_dir = out_dir / "videos" / CAMERA_KEY / "chunk-000"
    video_dir.mkdir(parents=True, exist_ok=True)
    video_path = video_dir / "file-000.mp4"
    write_video(frames, video_path, fps=int(FPS))

    # 5) Read actual FPS from encoded video
    video_meta = {}
    try:
        video_meta = iio.immeta(str(video_path))
    except Exception:
        video_meta = {}
    FPS_META = float(video_meta.get("fps", FPS))
    FPS_INT  = int(round(FPS_META))
    write_log(f"[INFO] Encoded video fps(meta)={video_meta.get('fps')} -> using FPS_META={FPS_META} (int={FPS_INT})")

    # 6) Write data parquet (timestamp per-episode from 0s)
    rows = []
    global_idx = 0
    for ep_idx, s, t in episodes:
        for local_i, g in enumerate(range(s, t)):
            rows.append({
                "action": np.asarray(actions[g], dtype=np.float32),
                "timestamp": np.float32(local_i / float(FPS_META)),
                "frame_index": np.int64(local_i),
                "episode_index": np.int64(ep_idx),
                "index": np.int64(global_idx),
                "task_index": np.int64(0)
            })
            global_idx += 1
    df = pd.DataFrame(rows)
    data_path = out_dir / "data" / "chunk-000" / "file-000.parquet"
    write_parquet(df, data_path)

    # 7) Write episodes parquet (seconds via actual fps)
    write_episodes_parquet(out_dir, episodes, len(frames), int(FPS_INT), CAMERA_KEY, video_duration_s=len(frames)/FPS_META)

    # 8) stats / tasks / info
    stats = compute_stats(actions)
    with (out_dir / "meta" / "stats.json").open("w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    write_tasks_parquet(out_dir, TASK_DESC)
    write_info_json(out_dir, CAMERA_KEY, H, W, int(FPS_INT), len(frames), total_episodes,
                    codec_str="h264", data_mb=100, video_mb=500)

    write_log(f"\n[OK] {out_dir}フォルダにLeRobot Dataset v3フォーマットで保存")
    
convert_button.on_click(convert)

## Push to Hugging Face Hub

In [27]:
from huggingface_hub import HfApi

upload_button = ipywidgets.Button(description='アップロード')
repo_name_widget = ipywidgets.Text(description='Repo Name')
def upload(change):
    api = HfApi()
    repo_id = repo_name = repo_name_widget.value
    write_log(f"アップロード開始 repo_id: {repo_id}...(時間がかかります)")
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True, private=False)
    api.upload_folder(
        repo_id=repo_id,
        repo_type="dataset",
        folder_path=str(out_dir),
        allow_patterns=["meta/**","data/**","videos/**","README.md"],
    )
    write_log(f"https://huggingface.co/datasets/{repo_id}")
    write_log(f"https://huggingface.co/spaces/lerobot/visualize_dataset?path={repo_id}")
upload_button.on_click(upload)

In [29]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.データセットを選択】</b> Huggingfaceにアップロードするデータセットを選択')
title2 = ipywidgets.HTML('<b>【2.LeRobot Datasetへ変換】</b> LeRobot Dataset v3へ変換')
title3 = ipywidgets.HTML('<b>【3.HuggingfaceにUpload】</b> Huggingfaceにアップロード')

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([dataset_dropdown,task_dropdown]),
    ipywidgets.HBox([xy_data_count_widget,speed_data_count_widget]),
    process_widget,
    title2,
    convert_button,
    process_widget,
    title3,
    ipywidgets.HBox([repo_name_widget, upload_button]),
    process_widget,
])
display(data_collection_widget)